# Fracture detection: FracAtlas + GRAZPEDWRI-DX (DenseNet121)

Trains the fracture / no-fracture classifier on two clinical radiograph datasets, with no leakage
between train, val and test (see `DATA_AUDIT.md`):
- **FracAtlas**: adult and paediatric hand, leg, hip and shoulder X-rays. It keeps **exactly the
  train/val/test split of run 1**, so its test results compare directly.
- **GRAZPEDWRI-DX**: paediatric wrist trauma X-rays from 6,091 patients, **split by patient**.
  Images with a cast, uncertain diagnoses and "fracture present but not visible in this view" images
  are removed.
- Exact and near duplicates are clustered across both datasets. Images with metal implants are kept
  out of training, because about 80% of them are fractured in both datasets.

The new model is compared with the deployed model and with run 1 on the same test images.

**Inputs (attached Kaggle datasets)**
- a code dataset containing the four modules, `run1_clean_manifest.csv`, and the baselines
  `baseline_densenet121_best.pth` (deployed model) and `baseline_run1_checkpoint.pt`
- [tommyngx/fracatlas](https://www.kaggle.com/datasets/tommyngx/fracatlas)
- [jasonroggy/grazpedwri-dx](https://www.kaggle.com/datasets/jasonroggy/grazpedwri-dx)

**Accelerator:** GPU T4. **Internet:** on (ImageNet weights).

**Outputs (`/kaggle/working`)**
- `weights/checkpoint_best.pt`: drop-in replacement for `Pipeline/Inference_Modules/models/fracture_detection/checkpoint_best.pt`
- `results.json`, `test_predictions.csv`, `clean_manifest.csv`, `figure_*.png`

In [ ]:
import glob
import json
import os
import sys
import tempfile
import time

os.environ.setdefault('TQDM_MININTERVAL', '30')   # keep progress output short in batch runs

INPUT_ROOT = os.environ.get('XCV_INPUT_ROOT', '/kaggle/input')
OUT_DIR = os.environ.get('XCV_OUT_DIR', '/kaggle/working')
SMOKE = os.environ.get('XCV_SMOKE') == '1'        # tiny CPU run used to test this notebook end to end

IMG_SIZE = 64 if SMOKE else 512
BATCH_SIZE = 4 if SMOKE else 24
NUM_EPOCHS = 1 if SMOKE else 20
NUM_WORKERS = 0 if SMOKE else 4
SEED = 42
PATIENCE = 5
WEIGHT_DECAY = 0.05
SOURCE_POWER = 0.5                    # a source's share of draws ~ n ** 0.5 (GRAZPEDWRI-DX is ~3x FracAtlas)
CLASS_BALANCE = 0.5                   # partial class balancing within each source (1 = fully balanced)
EXCLUDE_HARDWARE_FROM_TRAIN = True    # metal implants are ~80% fractured in both datasets: a shortcut
CACHE_MAX_SIDE = int(IMG_SIZE * 1.25)

SAVE_DIR = os.path.join(OUT_DIR, 'weights')
os.makedirs(SAVE_DIR, exist_ok=True)


def find_file(name, required=True):
    hits = sorted(glob.glob(os.path.join(INPUT_ROOT, '**', name), recursive=True))
    if not hits and required:
        raise FileNotFoundError(f'{name} not found under {INPUT_ROOT}; is the code dataset attached?')
    return hits[0] if hits else None


def find_root(required):
    """First directory under INPUT_ROOT that contains all `required` subdirectories."""
    for dirpath, _, _ in os.walk(INPUT_ROOT):
        if all(os.path.isdir(os.path.join(dirpath, r)) for r in required):
            return dirpath
    raise FileNotFoundError(f'No directory containing {required} under {INPUT_ROOT}')


def find_csv_root(column):
    """Directory of the first dataset.csv under INPUT_ROOT whose header contains `column`."""
    for p in sorted(glob.glob(os.path.join(INPUT_ROOT, '**', 'dataset.csv'), recursive=True)):
        with open(p, encoding='utf-8', errors='ignore') as f:
            if column in f.readline():
                return os.path.dirname(p)
    raise FileNotFoundError(f'No dataset.csv with a {column!r} column under {INPUT_ROOT}')


CODE_DIR = os.path.dirname(find_file('data_preprocessing.py'))
sys.path.insert(0, CODE_DIR)
ROOT_FRACATLAS = find_root(['images/Fractured', 'images/Non_fractured'])
ROOT_GRAZ = find_csv_root('filestem')
RUN1_MANIFEST = find_file('run1_clean_manifest.csv', required=False)
BASELINES = {name: path for name, path in [
    ('deployed model', find_file('baseline_densenet121_best.pth', required=False)),
    ('run 1 (A + FracAtlas, 384px)', find_file('baseline_run1_checkpoint.pt', required=False)),
] if path}
print('code          :', CODE_DIR)
print('FracAtlas     :', ROOT_FRACATLAS)
print('GRAZPEDWRI-DX :', ROOT_GRAZ)
print('run-1 split   :', RUN1_MANIFEST)
print('baselines     :', BASELINES)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader

from data_preprocessing import (IMAGENET_MEAN, IMAGENET_STD, XrayDataset, build_manifest, cache_resized,
                                clean_and_split, get_dataloaders, get_transforms, show_batch)
from model import build_densenet121, get_model_summary
from train import load_model, plot_training_history, predict_probs, train_model, tune_threshold
from evaluate_module2 import binary_metrics, evaluate_on_test

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
np.random.seed(SEED)
print('torch', torch.__version__, '| device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU')

# Also save every figure the modules show into OUT_DIR
_show = plt.show
_fig_count = [0]


def _show_and_save(*args, **kwargs):
    _fig_count[0] += 1
    plt.savefig(os.path.join(OUT_DIR, f'figure_{_fig_count[0]:02d}.png'), dpi=110, bbox_inches='tight')
    _show(*args, **kwargs)


plt.show = _show_and_save

## 1. Clean the data and build a split with no leakage
Fingerprints every image and merges exact and near copies across both datasets. It drops radiographs
that carry both labels, keeps one image per duplicate cluster, and assigns splits by group. A group
is a duplicate cluster plus, for GRAZPEDWRI-DX, every image of the same patient. FracAtlas groups
reuse their run-1 split. Kept images are then decoded once into a small 8-bit cache for fast epochs.

In [ ]:
t0 = time.time()
fixed_splits = None
if RUN1_MANIFEST:
    run1 = pd.read_csv(RUN1_MANIFEST, usecols=['source', 'name', 'split'])
    run1 = run1[run1['source'] == 'fracatlas']
    fixed_splits = dict(zip('fracatlas|' + run1['name'], run1['split']))
    print(f'Reusing the run-1 split for {len(fixed_splits)} FracAtlas images')

roots = {'fracatlas': ROOT_FRACATLAS, 'graz': ROOT_GRAZ}
files, thumbs = build_manifest(roots, cache_path=os.path.join(tempfile.gettempdir(), 'xcv_fingerprints.pkl'),
                               num_workers=1 if SMOKE else None)
manifest = clean_and_split(files, thumbs, seed=SEED, fixed_splits=fixed_splits, fixed_sources=['fracatlas'])
del thumbs

if RUN1_MANIFEST:
    r1_test = set(run1.loc[run1['split'] == 'test', 'name'])
    r2_test = set(manifest.loc[(manifest['source'] == 'fracatlas') & (manifest['split'] == 'test'), 'name'])
    print(f'FracAtlas test set identical to run 1: {r1_test == r2_test} ({len(r2_test)} images)')

if EXCLUDE_HARDWARE_FROM_TRAIN:
    drop = (manifest['split'] == 'train') & manifest['hardware'].eq(1)
    manifest = manifest[~drop].reset_index(drop=True)
    print(f'Dropped {int(drop.sum())} hardware images from train')

manifest = cache_resized(manifest, os.path.join(tempfile.gettempdir(), 'xcv_cache'), max_side=CACHE_MAX_SIDE,
                         num_workers=1 if SMOKE else None)
manifest.drop(columns=['phash']).to_csv(os.path.join(OUT_DIR, 'clean_manifest.csv'), index=False)
print(manifest.groupby(['split', 'source', 'label']).size().unstack(fill_value=0))
print(f'Data preparation took {time.time() - t0:.0f}s')

## 2. Data loaders

In [ ]:
loaders = get_dataloaders(manifest, img_size=IMG_SIZE, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          source_power=SOURCE_POWER, class_balance=CLASS_BALANCE)
for phase, dl in loaders.items():
    print(f'{phase:5s}: {len(dl.dataset)} images')
show_batch(loaders['train'])

## 3. Train
One epoch of head-only training, then the full network at 0.1x the head learning rate, with AdamW
weight decay 0.05. Model selection uses validation ROC-AUC, and the decision threshold is tuned on
validation and stored in the checkpoint.

In [ ]:
model = build_densenet121(num_classes=2, pretrained=not SMOKE, dropout_rate=0.5)
get_model_summary(model, input_size=(1, 3, IMG_SIZE, IMG_SIZE))
history = train_model(model, loaders['train'], loaders['val'],
                      num_epochs=NUM_EPOCHS, device=DEVICE, save_dir=SAVE_DIR, patience=PATIENCE,
                      lr=1e-3, backbone_lr_mult=0.1, weight_decay=WEIGHT_DECAY, freeze_epochs=1,
                      label_smoothing=0.05, threshold_mode='youden',
                      checkpoint_meta={'data': 'FracAtlas + GRAZPEDWRI-DX, deduplicated, patient-level split',
                                       'seed': SEED})
plot_training_history(history)

## 4. Test-set evaluation
Each dataset is a separate benchmark, with metal-implant images reported as their own group.

In [ ]:
best_model = load_model(os.path.join(SAVE_DIR, 'checkpoint_best.pt'), device=DEVICE)
test_df = manifest[manifest['split'] == 'test'].reset_index(drop=True)
has_hw = test_df['hardware'].eq(1).values
groups = np.where(has_hw, test_df['source'] + '+hardware', test_df['source'])
res = evaluate_on_test(best_model, loaders['test'], DEVICE, sources=groups)

# Screening operating point: the highest threshold that reaches 90% sensitivity on validation
val_probs, val_labels, _ = predict_probs(best_model, loaders['val'], DEVICE, tta=True)
thr_sens90 = tune_threshold(val_labels, val_probs, mode='sensitivity', target_sensitivity=0.90)

BENCHMARKS = {
    'FracAtlas': (test_df['source'] == 'fracatlas').values & ~has_hw,
    'GRAZPEDWRI-DX': (test_df['source'] == 'graz').values & ~has_hw,
    'all test images': np.ones(len(test_df), bool),
}


def bench_table(probs, labels, thr):
    return pd.DataFrame({b: binary_metrics(labels[m], probs[m], thr) for b, m in BENCHMARKS.items()}).T


print(f"New model at the val-tuned threshold {res['threshold']:.3f}:")
print(bench_table(res['probs'], res['labels'], res['threshold']).round(4).to_string())
print(f'\nNew model at the sens-90 threshold {thr_sens90:.3f}:')
print(bench_table(res['probs'], res['labels'], thr_sens90).round(4).to_string())

## 5. Comparison on the same test images
Each model uses its own preprocessing and threshold: the deployed model resizes to 256, centre-crops
to 224 and uses 0.6, and run 1 uses its checkpoint settings.
- **FracAtlas:** run 1 used the same FracAtlas train split, so this is like-for-like. The deployed model
  trained on a random 70% of FracAtlas, so its FracAtlas numbers are optimistic.
- **GRAZPEDWRI-DX:** neither baseline has ever seen it, so this column is a clean comparison.

In [ ]:
legacy_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
model_probs = {'run 2 (this run)': (res['probs'], res['threshold'])}
for name, path in BASELINES.items():
    m = load_model(path, device=DEVICE)
    meta = m.fracture_meta
    if meta.get('preprocessing') == 'pad_to_square':
        tf, thr, tta = get_transforms('test', int(meta['img_size'])), float(meta['threshold']), bool(meta.get('tta', True))
    else:
        tf, thr, tta = legacy_tf, 0.6, False   # the deployed pipeline's settings
    dl = DataLoader(XrayDataset(test_df['path'].values, test_df['label'].values, transform=tf),
                    batch_size=32, num_workers=NUM_WORKERS)
    p, _, _ = predict_probs(m, dl, DEVICE, tta=tta)
    model_probs[name] = (p, thr)
    del m
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

labels = res['labels']
comparison = {}
for bench, mask in BENCHMARKS.items():
    comparison[bench] = pd.DataFrame({name: binary_metrics(labels[mask], p[mask], thr)
                                      for name, (p, thr) in model_probs.items()}).T
    print(f'\n=== {bench} test (n={int(mask.sum())}, fractured={int(labels[mask].sum())}) ===')
    print(comparison[bench][['auc', 'accuracy', 'balanced_acc', 'sensitivity', 'specificity',
                             'precision', 'f1']].round(4).to_string())

## 6. Save results

In [ ]:
preds = test_df[['orig_path', 'source', 'hardware', 'label']].copy()
for name, (p, _) in model_probs.items():
    preds[f'prob {name}'] = p
preds.to_csv(os.path.join(OUT_DIR, 'test_predictions.csv'), index=False)

results = {
    'config': {'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE, 'num_epochs': NUM_EPOCHS, 'seed': SEED,
               'patience': PATIENCE, 'weight_decay': WEIGHT_DECAY, 'source_power': SOURCE_POWER,
               'class_balance': CLASS_BALANCE, 'exclude_hardware_from_train': EXCLUDE_HARDWARE_FROM_TRAIN},
    'split_sizes': {'/'.join(map(str, k)): int(v)
                    for k, v in manifest.groupby(['split', 'source', 'label']).size().items()},
    'threshold_youden': res['threshold'],
    'threshold_sens90': thr_sens90,
    'new_model_per_source': res['per_source'].to_dict(orient='index'),
    'new_model_at_sens90': bench_table(res['probs'], res['labels'], thr_sens90).to_dict(orient='index'),
    'comparison': {b: t.to_dict(orient='index') for b, t in comparison.items()},
    'model_thresholds': {name: thr for name, (_, thr) in model_probs.items()},
    'history': history,
}
with open(os.path.join(OUT_DIR, 'results.json'), 'w') as f:
    json.dump(results, f, indent=2, default=float)

# checkpoint_best.pt is what the Pipeline loads; the last-epoch weights only bloat the output
os.remove(os.path.join(SAVE_DIR, 'densenet121_last.pth'))
print(sorted(os.listdir(OUT_DIR)), sorted(os.listdir(SAVE_DIR)))